# Chapter 10 — Mixture of Experts and Routing

Interactive theory notebook: router computation, auxiliary losses, load balancing,
expert capacity, MoE forward pass, specialisation measurement, communication cost,
routing analysis, parameter counting, and scaling comparisons.

In [ ]:
# === Setup ===
import numpy as np
np.random.seed(42)

try:
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    HAS_MPL = True
except ImportError:
    HAS_MPL = False

def fmt_vec(v, decimals=4):
    return "[" + ", ".join(f"{x:.{decimals}f}" for x in v) + "]"

def fmt_sci(x):
    if abs(x) >= 1e9: return f"{x/1e9:.2f}B"
    if abs(x) >= 1e6: return f"{x/1e6:.2f}M"
    if abs(x) >= 1e3: return f"{x/1e3:.1f}K"
    return f"{x:.2f}"

def fmt_bytes(b):
    if b >= 1e12: return f"{b/1e12:.2f} TB"
    if b >= 1e9: return f"{b/1e9:.2f} GB"
    if b >= 1e6: return f"{b/1e6:.2f} MB"
    if b >= 1e3: return f"{b/1e3:.1f} KB"
    return f"{b:.0f} B"

def print_table(headers, rows, col_width=16):
    hdr = " | ".join(h.center(col_width) for h in headers)
    print(hdr)
    print("-" * len(hdr))
    for row in rows:
        print(" | ".join(str(c).center(col_width) for c in row))

def softmax(z):
    z = np.asarray(z, dtype=np.float64)
    z = z - z.max()
    e = np.exp(z)
    return e / e.sum()

print("Setup complete ✓")

---
## §1 Router Computation and Top-k Selection

The router computes logits $z = W_g x$, applies softmax to get gate probabilities,
selects top-k experts, and renormalises the selected gates.

In [ ]:
# === §1: Router Computation and Top-k Selection ===
np.random.seed(42)

d_model = 6       # embedding dimension
N_experts = 8     # total experts
k = 2             # top-k

# --- 1a: Compute router logits and probabilities ---
W_g = np.random.randn(N_experts, d_model) * 0.1  # router weight matrix
x = np.random.randn(d_model)                      # single token embedding

logits = W_g @ x
probs = softmax(logits)

print("Router logits z = W_g x:")
for i in range(N_experts):
    print(f"  Expert {i}: logit = {logits[i]:+.4f}  prob = {probs[i]:.4f}")

# --- 1b: Top-k selection ---
top_k_indices = np.argsort(probs)[-k:][::-1]
top_k_probs = probs[top_k_indices]

print(f"\nTop-{k} experts: {top_k_indices}")
print(f"Top-{k} probabilities (raw): {fmt_vec(top_k_probs)}")

# --- 1c: Renormalise ---
renorm_gates = top_k_probs / top_k_probs.sum()
print(f"Renormalised gates: {fmt_vec(renorm_gates)}")
print(f"Sum of renormalised gates: {renorm_gates.sum():.6f}")

# --- 1d: Simulate expert outputs and weighted combination ---
d_out = 4
expert_outputs = {}
for idx in top_k_indices:
    W1 = np.random.randn(8, d_model) * 0.1
    W2 = np.random.randn(d_out, 8) * 0.1
    expert_outputs[idx] = W2 @ np.maximum(0, W1 @ x)  # ReLU FFN

moe_output = sum(renorm_gates[i] * expert_outputs[top_k_indices[i]] for i in range(k))
print(f"\nExpert {top_k_indices[0]} output: {fmt_vec(expert_outputs[top_k_indices[0]])}")
print(f"Expert {top_k_indices[1]} output: {fmt_vec(expert_outputs[top_k_indices[1]])}")
print(f"Weighted MoE output:         {fmt_vec(moe_output)}")

---
## §2 Auxiliary Loss Calculation

The auxiliary loss $\mathcal{L}_{\text{aux}} = \alpha \sum_i f_i \cdot P_i$ penalises
imbalanced routing. $f_i$ = fraction of tokens dispatched to expert $i$;
$P_i$ = mean gate probability for expert $i$.

In [ ]:
# === §2: Auxiliary Loss Calculation ===
np.random.seed(42)

T = 64        # tokens in batch
N = 8         # experts
k = 2         # top-k
alpha = 0.01  # aux loss coefficient

# --- 2a: Simulate routing for a batch ---
W_g = np.random.randn(N, 16) * 0.1
X = np.random.randn(T, 16)

all_logits = X @ W_g.T
all_probs = np.array([softmax(z) for z in all_logits])  # (T, N)

# Compute f_i: fraction of tokens dispatched to each expert
dispatch_count = np.zeros(N)
for t in range(T):
    top_k_idx = np.argsort(all_probs[t])[-k:][::-1]
    dispatch_count[top_k_idx] += 1

f = dispatch_count / T
P = all_probs.mean(axis=0)  # mean gate probability per expert

print("Expert load fractions f_i:")
for i in range(N):
    bar = "█" * int(f[i] * 40)
    print(f"  Expert {i}: f={f[i]:.3f}  P={P[i]:.4f}  {bar}")

# --- 2b: Compute auxiliary loss ---
L_aux = alpha * N * np.sum(f * P)  # scaled by N to make gradient independent of N
uniform_f = np.ones(N) / N * k  # uniform: each expert gets k*T/N dispatches / T
uniform_P = np.ones(N) / N
L_aux_uniform = alpha * N * np.sum(uniform_f * uniform_P)

print(f"\nAuxiliary loss (actual):  {L_aux:.6f}")
print(f"Auxiliary loss (uniform): {L_aux_uniform:.6f}")
print(f"Ratio (actual/uniform):  {L_aux/L_aux_uniform:.4f}")
print(f"\nActual > uniform → penalty for imbalance: {L_aux > L_aux_uniform}")

---
## §3 Load Balancing Simulation

Simulate how different auxiliary loss weights $\alpha$ affect expert load distribution
over training, and observe the expert collapse phenomenon.

In [ ]:
# === §3: Load Balancing Simulation ===
np.random.seed(42)

N = 4
d = 8
T = 32
k_top = 1
lr = 0.05
steps = 200

def simulate_training(alpha_val, steps=200):
    """Simulate routing updates with given aux loss weight."""
    np.random.seed(42)
    W = np.random.randn(N, d) * 0.01
    load_history = []
    
    for step in range(steps):
        X = np.random.randn(T, d)
        logits = X @ W.T
        probs = np.array([softmax(z) for z in logits])
        
        # Count dispatches
        counts = np.zeros(N)
        for t in range(T):
            top_idx = np.argmax(probs[t])
            counts[top_idx] += 1
        f = counts / T
        P = probs.mean(axis=0)
        load_history.append(f.copy())
        
        # Gradient for router: task gradient (random for simulation) + aux gradient
        task_grad = np.random.randn(N, d) * 0.001
        # Aux loss gradient: penalise high f_i * P_i
        aux_grad = np.zeros((N, d))
        for i in range(N):
            aux_grad[i] = alpha_val * N * f[i] * (X.T @ probs[:, i]) / T
        
        # Simulate "rich-get-richer" bias: popular experts get better task gradient
        popularity_bias = np.outer(f, np.ones(d)) * 0.01
        W -= lr * (task_grad + popularity_bias - aux_grad)
    
    return np.array(load_history)

# --- 3a: Compare different alpha values ---
alphas = [0.0, 0.001, 0.01, 0.1]
results = {}
for a in alphas:
    results[a] = simulate_training(a, steps)

print("Final expert load distribution (f_i) after 200 steps:")
print_table(
    ["Alpha"] + [f"Expert {i}" for i in range(N)] + ["Max/Min Ratio"],
    [
        [a] + [f"{results[a][-1, i]:.3f}" for i in range(N)] + 
        [f"{results[a][-1].max() / max(results[a][-1].min(), 1e-10):.1f}"]
        for a in alphas
    ]
)

# --- 3b: Visualise load evolution ---
if HAS_MPL:
    fig, axes = plt.subplots(1, 4, figsize=(16, 3), sharey=True)
    for ax, a in zip(axes, alphas):
        for i in range(N):
            ax.plot(results[a][:, i], label=f"Expert {i}")
        ax.axhline(y=1/N, color="k", linestyle="--", alpha=0.5, label="uniform")
        ax.set_title(f"α = {a}")
        ax.set_xlabel("Step")
        ax.set_ylim(0, 1)
        if a == 0.0:
            ax.set_ylabel("Expert load f_i")
    axes[-1].legend(fontsize=7, loc="upper right")
    plt.suptitle("Expert Load vs Auxiliary Loss Weight")
    plt.tight_layout()
    plt.savefig("load_balancing_simulation.png", dpi=100, bbox_inches="tight")
    plt.show()
    print("Saved: load_balancing_simulation.png")
else:
    print("(matplotlib unavailable — skipping plot)")

print("\nKey insight: α=0 → collapse (one expert dominates); α=0.01–0.1 → balanced load")

---
## §4 Expert Capacity and Token Dropping

Each expert has a maximum capacity $C$. Tokens exceeding this limit are dropped
or handled via fallback routing. Capacity factor $CF$ controls the slack.

In [ ]:
# === §4: Expert Capacity and Token Dropping ===
np.random.seed(42)

T = 1024      # tokens
N = 8         # experts
k = 2         # top-k

# --- 4a: Capacity computation ---
print("Capacity for different capacity factors (CF):")
print_table(
    ["CF", "C = ceil(CF*T*k/N)", "Total slots", "Overhead %"],
    [
        [
            f"{cf:.2f}",
            int(np.ceil(cf * T * k / N)),
            int(np.ceil(cf * T * k / N)) * N,
            f"{(int(np.ceil(cf * T * k / N)) * N - T * k) / (T * k) * 100:.1f}%"
        ]
        for cf in [1.0, 1.1, 1.25, 1.5, 2.0]
    ]
)

# --- 4b: Simulate token assignment and dropping ---
CF = 1.25
C = int(np.ceil(CF * T * k / N))
print(f"\nSimulation: T={T}, N={N}, k={k}, CF={CF}, Capacity C={C}")

# Simulate routing with unbalanced load
W_g = np.random.randn(N, 32) * 0.1
# Bias some experts to be more popular
W_g[0] += 0.3  # expert 0 biased
W_g[1] += 0.15  # expert 1 biased

X = np.random.randn(T, 32)
logits = X @ W_g.T
probs = np.array([softmax(z) for z in logits])

# Route tokens
expert_buffers = {i: [] for i in range(N)}
dropped_tokens = []

for t in range(T):
    top_indices = np.argsort(probs[t])[-k:][::-1]
    assigned = False
    for idx in top_indices:
        if len(expert_buffers[idx]) < C:
            expert_buffers[idx].append(t)
            assigned = True
        # If overflow: token dropped from this expert
    if not assigned:
        dropped_tokens.append(t)

print(f"\nExpert load after capacity-limited routing:")
total_assigned = 0
for i in range(N):
    count = len(expert_buffers[i])
    total_assigned += count
    bar = "█" * (count // 8)
    overflow = " *** OVERFLOW ***" if count >= C else ""
    print(f"  Expert {i}: {count:4d}/{C} tokens  {bar}{overflow}")

print(f"\nDropped tokens (not assigned to ANY expert): {len(dropped_tokens)}")
print(f"Drop rate: {len(dropped_tokens)/T*100:.1f}%")
print(f"Total assignments (token-expert pairs): {total_assigned} / {T*k} possible")

---
## §5 MoE Forward Pass Simulation

Complete end-to-end MoE forward pass: embedding → router → dispatch → expert compute → combine.

In [ ]:
# === §5: MoE Forward Pass Simulation ===
np.random.seed(42)

d_model = 16
d_ff = 32
N = 4         # experts
k = 2         # top-k
T = 8         # tokens in batch

# --- 5a: Initialise experts and router ---
class Expert:
    def __init__(self, d_in, d_ff):
        self.W1 = np.random.randn(d_ff, d_in) * np.sqrt(2.0 / d_in)
        self.b1 = np.zeros(d_ff)
        self.W2 = np.random.randn(d_in, d_ff) * np.sqrt(2.0 / d_ff)
        self.b2 = np.zeros(d_in)
    
    def forward(self, x):
        h = np.maximum(0, self.W1 @ x + self.b1)  # ReLU
        return self.W2 @ h + self.b2

experts = [Expert(d_model, d_ff) for _ in range(N)]
W_router = np.random.randn(N, d_model) * 0.1

# --- 5b: Full MoE forward pass ---
X = np.random.randn(T, d_model)  # batch of tokens

outputs = np.zeros((T, d_model))
routing_decisions = []

for t in range(T):
    x = X[t]
    
    # Router
    logits = W_router @ x
    probs = softmax(logits)
    
    # Top-k selection
    top_idx = np.argsort(probs)[-k:][::-1]
    top_probs = probs[top_idx]
    gates = top_probs / top_probs.sum()  # renormalise
    
    routing_decisions.append(top_idx.copy())
    
    # Expert computation + weighted combination
    y = np.zeros(d_model)
    for i, idx in enumerate(top_idx):
        y += gates[i] * experts[idx].forward(x)
    
    # Residual connection
    outputs[t] = x + y

print("MoE Forward Pass Results:")
print(f"  Input shape:  ({T}, {d_model})")
print(f"  Output shape: ({T}, {d_model})")
print(f"  Experts: {N}, Top-k: {k}")

print(f"\nRouting decisions (expert indices per token):")
expert_counts = np.zeros(N)
for t, rd in enumerate(routing_decisions):
    print(f"  Token {t}: experts {rd}")
    for idx in rd:
        expert_counts[idx] += 1

print(f"\nExpert utilisation:")
for i in range(N):
    print(f"  Expert {i}: {int(expert_counts[i]):2d} assignments ({expert_counts[i]/(T*k)*100:.0f}%)")

# --- 5c: Compare MoE vs Dense forward pass ---
dense_ffn = Expert(d_model, d_ff * k)  # Dense FFN with same active FLOPs
dense_outputs = np.array([x + dense_ffn.forward(x) for x in X])

moe_norms = np.linalg.norm(outputs, axis=1)
dense_norms = np.linalg.norm(dense_outputs, axis=1)
print(f"\nOutput L2 norms (MoE vs Dense):")
print(f"  MoE mean:   {moe_norms.mean():.4f} ± {moe_norms.std():.4f}")
print(f"  Dense mean: {dense_norms.mean():.4f} ± {dense_norms.std():.4f}")

---
## §6 Expert Specialisation Measurement

Analyse whether experts specialise on different input types by routing
synthetic data from distinct clusters and measuring specialisation entropy.

In [ ]:
# === §6: Expert Specialisation Measurement ===
np.random.seed(42)

d = 16
N = 8
k = 2
n_clusters = 4  # synthetic input "domains"
tokens_per_cluster = 200
T_total = n_clusters * tokens_per_cluster

# --- 6a: Create synthetic clustered data ---
cluster_centers = np.random.randn(n_clusters, d) * 3
cluster_labels = []
all_tokens = []

for c in range(n_clusters):
    tokens = cluster_centers[c] + np.random.randn(tokens_per_cluster, d) * 0.5
    all_tokens.append(tokens)
    cluster_labels.extend([c] * tokens_per_cluster)

X = np.vstack(all_tokens)
cluster_labels = np.array(cluster_labels)

# Shuffle
perm = np.random.permutation(T_total)
X = X[perm]
cluster_labels = cluster_labels[perm]

# --- 6b: Simulate learning a router that captures cluster structure ---
# Use cluster-aware router (simulates post-training specialisation)
W_g = np.random.randn(N, d) * 0.01
# Add some cluster-correlated structure
for i in range(N):
    cluster_idx = i % n_clusters
    W_g[i] += cluster_centers[cluster_idx] * 0.1

# Route all tokens
expert_cluster_counts = np.zeros((N, n_clusters))  # expert × cluster

logits = X @ W_g.T
for t in range(T_total):
    probs = softmax(logits[t])
    top_idx = np.argsort(probs)[-k:][::-1]
    for idx in top_idx:
        expert_cluster_counts[idx, cluster_labels[t]] += 1

# --- 6c: Compute specialisation entropy per expert ---
print("Expert × Cluster routing matrix (normalised per expert):")
headers = ["Expert"] + [f"Cluster {c}" for c in range(n_clusters)] + ["Entropy", "Status"]
rows = []
for i in range(N):
    total = expert_cluster_counts[i].sum()
    if total == 0:
        rows.append([f"Expert {i}"] + ["0.000"] * n_clusters + ["N/A", "DEAD"])
        continue
    dist = expert_cluster_counts[i] / total
    entropy = -np.sum(dist[dist > 0] * np.log2(dist[dist > 0]))
    max_entropy = np.log2(n_clusters)
    status = "Specialised" if entropy < max_entropy * 0.7 else "Generalist"
    rows.append(
        [f"Expert {i}"] + [f"{d:.3f}" for d in dist] + [f"{entropy:.3f}", status]
    )

print_table(headers, rows, col_width=12)

max_ent = np.log2(n_clusters)
print(f"\nMax possible entropy (uniform across {n_clusters} clusters): {max_ent:.3f} bits")
print(f"Specialised = entropy < {max_ent * 0.7:.3f}")

---
## §7 All-to-All Communication Cost

Compute the communication overhead for expert parallelism: each GPU sends tokens
to every other GPU holding the target expert.

In [ ]:
# === §7: All-to-All Communication Cost ===
np.random.seed(42)

# --- 7a: Communication math for expert parallelism ---
configs = [
    {"name": "Small",  "EP": 8,  "T": 2048,  "d": 2048, "bytes": 2, "bw_gbps": 600},
    {"name": "Medium", "EP": 32, "T": 4096,  "d": 4096, "bytes": 2, "bw_gbps": 600},
    {"name": "Large",  "EP": 64, "T": 4096,  "d": 4096, "bytes": 2, "bw_gbps": 400},
    {"name": "V3",     "EP": 64, "T": 8192,  "d": 7168, "bytes": 2, "bw_gbps": 400},
]

print("All-to-All Communication Cost (per MoE layer, one direction):")
headers = ["Config", "EP", "Tokens", "d_model", "Data/GPU", "Time (ms)", "×2 dirs"]
rows = []
for c in configs:
    # Each GPU holds T/EP tokens; sends to EP-1 other GPUs
    # Total data sent per GPU: (T/EP) * d * bytes * (EP-1)/EP ≈ T * d * bytes * (1 - 1/EP)
    tokens_per_gpu = c["T"] / c["EP"]
    data_per_gpu = tokens_per_gpu * c["d"] * c["bytes"] * (c["EP"] - 1) / c["EP"]
    time_ms = data_per_gpu / (c["bw_gbps"] * 1e9) * 1000
    rows.append([
        c["name"], c["EP"], c["T"], c["d"],
        fmt_bytes(data_per_gpu), f"{time_ms:.3f}", f"{2*time_ms:.3f}"
    ])

print_table(headers, rows, col_width=12)

# --- 7b: Communication vs compute ratio ---
print("\n\nCommunication vs Expert Compute (per layer):")
d = 4096
d_ff = 14336
k = 2
T = 4096
EP = 32
bytes_per_elem = 2  # BF16

# Expert compute FLOPs per GPU
experts_per_gpu = 8 // EP if 8 > EP else 1  # simplified
tokens_per_expert = T * k // 8  # uniform
expert_flops = 2 * d * d_ff * tokens_per_expert  # one expert, forward
gpu_tflops = 312  # H100 BF16
compute_time_ms = expert_flops / (gpu_tflops * 1e12) * 1000

# Communication
comm_data = (T / EP) * d * bytes_per_elem
bw = 600e9  # NVLink
comm_time_ms = 2 * comm_data / bw * 1000  # dispatch + combine

print(f"  Expert compute per GPU: {expert_flops/1e9:.2f} GFLOPs → {compute_time_ms:.3f} ms")
print(f"  Communication (2×): {fmt_bytes(2*comm_data)} → {comm_time_ms:.3f} ms")
print(f"  Comm/compute ratio: {comm_time_ms/max(compute_time_ms,1e-10):.2f}")
if comm_time_ms < compute_time_ms:
    print("  → Communication fully overlappable with compute ✓")
else:
    print("  → Communication dominant — need compute-comm overlap strategies")

---
## §8 Routing Analysis — Gini Coefficient and Entropy

Quantify routing quality via Gini coefficient (load balance) and
routing entropy (confidence/diversity).

In [ ]:
# === §8: Routing Analysis — Gini Coefficient and Entropy ===
np.random.seed(42)

def gini_coefficient(load):
    """Compute Gini coefficient for expert load distribution."""
    load = np.array(load, dtype=float)
    n = len(load)
    if load.sum() == 0:
        return 0.0
    diff_sum = sum(abs(load[i] - load[j]) for i in range(n) for j in range(n))
    return diff_sum / (2 * n * load.sum())

def routing_entropy(probs):
    """Compute entropy of routing distribution."""
    p = np.array(probs)
    p = p[p > 0]
    return -np.sum(p * np.log2(p))

# --- 8a: Gini coefficient for different load distributions ---
load_scenarios = {
    "Perfect uniform":     [0.125, 0.125, 0.125, 0.125, 0.125, 0.125, 0.125, 0.125],
    "Mild imbalance":      [0.18, 0.16, 0.14, 0.13, 0.12, 0.11, 0.09, 0.07],
    "Moderate imbalance":  [0.30, 0.20, 0.15, 0.12, 0.10, 0.07, 0.04, 0.02],
    "Severe imbalance":    [0.50, 0.25, 0.10, 0.08, 0.04, 0.02, 0.01, 0.00],
    "Expert collapse":     [0.90, 0.05, 0.03, 0.02, 0.00, 0.00, 0.00, 0.00],
}

print("Gini Coefficient Analysis:")
print_table(
    ["Scenario", "Gini", "Max Load", "Dead Experts", "Health"],
    [
        [
            name,
            f"{gini_coefficient(load):.4f}",
            f"{max(load):.3f}",
            sum(1 for l in load if l < 0.01),
            "✓ Healthy" if gini_coefficient(load) < 0.15 else
            "⚠ Warning" if gini_coefficient(load) < 0.35 else "✗ Critical"
        ]
        for name, load in load_scenarios.items()
    ],
    col_width=18
)

# --- 8b: Routing entropy per token ---
N = 8
print(f"\nRouting Entropy Analysis (N={N} experts):")
print(f"Max entropy (uniform): {np.log2(N):.3f} bits\n")

entropy_scenarios = {
    "Uniform":        [1/N] * N,
    "Mildly peaked":  [0.25, 0.20, 0.15, 0.12, 0.10, 0.08, 0.06, 0.04],
    "Confident":      [0.60, 0.25, 0.10, 0.03, 0.01, 0.005, 0.003, 0.002],
    "Near one-hot":   [0.95, 0.03, 0.01, 0.005, 0.002, 0.001, 0.001, 0.001],
}

print_table(
    ["Distribution", "Entropy (bits)", "Fraction of max", "Interpretation"],
    [
        [
            name,
            f"{routing_entropy(probs):.3f}",
            f"{routing_entropy(probs)/np.log2(N)*100:.1f}%",
            "Maximum diversity" if routing_entropy(probs) > 2.5 else
            "Good diversity" if routing_entropy(probs) > 1.5 else
            "Moderately confident" if routing_entropy(probs) > 0.5 else "Highly confident"
        ]
        for name, probs in entropy_scenarios.items()
    ],
    col_width=18
)

---
## §9 Parameter Counting — Dense vs MoE

Count total and active parameters for dense and MoE models to quantify
the capacity-compute separation.

In [ ]:
# === §9: Parameter Counting — Dense vs MoE ===
np.random.seed(42)

def count_params(d_model, d_ff, n_heads, d_head, n_layers, n_experts=1, k=1,
                 n_shared=0, vocab_size=32000):
    """Count parameters for transformer with MoE."""
    # Attention params per layer
    attn = 4 * d_model * (n_heads * d_head)  # Q, K, V, O projections
    
    # FFN params per expert
    ffn_per_expert = 2 * d_model * d_ff + d_model + d_ff  # W1, W2, biases
    
    # Router params per layer
    router = n_experts * d_model if n_experts > 1 else 0
    
    # Total FFN params per layer
    ffn_total = n_experts * ffn_per_expert + n_shared * ffn_per_expert
    ffn_active = k * ffn_per_expert + n_shared * ffn_per_expert
    
    # Layer norms (2 per layer)
    ln = 4 * d_model
    
    # Per layer
    layer_total = attn + ffn_total + router + ln
    layer_active = attn + ffn_active + router + ln
    
    # Embedding
    embed = vocab_size * d_model
    
    total = n_layers * layer_total + embed
    active = n_layers * layer_active + embed
    
    return {
        "total": total, "active": active,
        "attn_per_layer": attn, "ffn_total_per_layer": ffn_total,
        "ffn_active_per_layer": ffn_active,
        "router_per_layer": router, "embed": embed,
        "sparsity_ratio": active / total,
    }

# --- 9a: Compare architectures ---
models = [
    ("LLaMA-2 7B (Dense)",   {"d_model": 4096, "d_ff": 11008, "n_heads": 32, "d_head": 128,
                              "n_layers": 32, "n_experts": 1, "k": 1}),
    ("Mixtral 8×7B",         {"d_model": 4096, "d_ff": 14336, "n_heads": 32, "d_head": 128,
                              "n_layers": 32, "n_experts": 8, "k": 2}),
    ("Dense 46B equiv",      {"d_model": 8192, "d_ff": 28672, "n_heads": 64, "d_head": 128,
                              "n_layers": 40, "n_experts": 1, "k": 1}),
    ("DeepSeek-V2 style",    {"d_model": 5120, "d_ff": 1536,  "n_heads": 40,  "d_head": 128,
                              "n_layers": 60, "n_experts": 160, "k": 6, "n_shared": 2}),
    ("DeepSeek-V3 style",    {"d_model": 7168, "d_ff": 2048,  "n_heads": 56,  "d_head": 128,
                              "n_layers": 61, "n_experts": 256, "k": 8, "n_shared": 1}),
]

print("Model Parameter Comparison:")
headers = ["Model", "Total Params", "Active Params", "Active %", "Experts", "Top-k"]
rows = []
for name, cfg in models:
    p = count_params(**cfg, vocab_size=32000)
    rows.append([
        name, fmt_sci(p["total"]), fmt_sci(p["active"]),
        f"{p['sparsity_ratio']*100:.1f}%",
        cfg["n_experts"], cfg["k"]
    ])

print_table(headers, rows, col_width=16)

# --- 9b: Memory requirements ---
print("\n\nMemory Requirements (BF16 — 2 bytes/param):")
headers = ["Model", "Total Memory", "Active Memory", "Memory Overhead"]
rows = []
for name, cfg in models:
    p = count_params(**cfg, vocab_size=32000)
    total_mem = p["total"] * 2
    active_mem = p["active"] * 2
    rows.append([
        name, fmt_bytes(total_mem), fmt_bytes(active_mem),
        f"{total_mem/active_mem:.1f}×"
    ])

print_table(headers, rows, col_width=16)

---
## §10 Scaling Comparison and Z-Loss

Compare quality scaling of MoE vs dense models, and demonstrate Z-loss regularisation
on router logits.

In [ ]:
# === §10: Scaling Comparison and Z-Loss ===
np.random.seed(42)

# --- 10a: MoE vs Dense scaling laws ---
# Simplified scaling: loss ∝ C^{-α} where C = effective compute/capacity
# Dense: L(N) = a * N^{-0.076}   (Chinchilla-like)
# MoE:   L(N_active, N_total) ≈ a * N_active^{-0.076} * (1 + β*log(N_total/N_active))

a = 5.0
alpha_exp = 0.076
beta = 0.02

active_params = np.array([1e9, 3e9, 7e9, 13e9, 30e9, 70e9])
expert_multipliers = [1, 4, 8, 16, 64]  # total/active ratio

print("Quality Estimates (lower loss = better):")
headers = ["Active Params"] + [f"{m}× total" for m in expert_multipliers]
rows = []
for N_act in active_params:
    row = [fmt_sci(N_act)]
    for mult in expert_multipliers:
        N_total = N_act * mult
        loss = a * N_act**(-alpha_exp) * (1 - beta * np.log(max(mult, 1)))
        row.append(f"{loss:.4f}")
    rows.append(row)

print_table(headers, rows, col_width=14)

print("\nDiminishing returns: going from 1× to 8× helps more than 8× to 64×")

# --- 10b: Z-Loss computation ---
print("\n" + "="*60)
print("Z-Loss Regularisation")
print("="*60)

# Z-loss penalises large router logits
def z_loss(logits):
    """L_z = (log(sum(exp(z_i))))^2"""
    z = np.array(logits, dtype=float)
    logsumexp = np.log(np.sum(np.exp(z - z.max())) ) + z.max()  # numerically stable
    return logsumexp ** 2

logit_scenarios = [
    ("Moderate logits",   [1.2, 0.8, -0.3, 0.5]),
    ("Large logits",      [3.2, 1.1, -0.5, 2.8]),
    ("Very large logits", [8.5, 2.0, -1.0, 7.2]),
    ("Near-uniform",      [1.0, 1.0, 1.0, 1.0]),
    ("All zeros",         [0.0, 0.0, 0.0, 0.0]),
]

print("\nZ-Loss for different router logit magnitudes:")
headers = ["Scenario", "Logits", "Softmax", "Z-Loss", "Effect"]
rows = []
for name, z in logit_scenarios:
    probs = softmax(z)
    zl = z_loss(z)
    max_prob = max(probs)
    effect = "Low penalty" if zl < 5 else "Moderate" if zl < 20 else "Strong penalty"
    rows.append([
        name,
        fmt_vec(z, 1),
        fmt_vec(probs, 3),
        f"{zl:.3f}",
        effect
    ])

print_table(headers, rows, col_width=22)

# --- 10c: Visualise Z-loss landscape ---
if HAS_MPL:
    scales = np.linspace(0.1, 10, 100)
    z_losses = [z_loss([s, 0.5*s, -0.2*s, 0.8*s]) for s in scales]
    
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.plot(scales, z_losses, 'b-', linewidth=2)
    ax.set_xlabel("Logit Scale Factor")
    ax.set_ylabel("Z-Loss")
    ax.set_title("Z-Loss vs Router Logit Magnitude")
    ax.axhline(y=5, color='orange', linestyle='--', alpha=0.7, label='Moderate threshold')
    ax.axhline(y=20, color='red', linestyle='--', alpha=0.7, label='Strong threshold')
    ax.legend()
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig("z_loss_landscape.png", dpi=100, bbox_inches="tight")
    plt.show()
    print("Saved: z_loss_landscape.png")
else:
    print("(matplotlib unavailable — skipping plot)")

print("\n" + "="*60)
print("THEORY NOTEBOOK COMPLETE")
print("="*60)
print("\nKey Takeaways:")
print("  1. Top-k routing selects k experts; renormalises gate weights")
print("  2. Auxiliary loss L_aux = α·N·Σ(f_i·P_i) prevents expert collapse")
print("  3. α=0 → collapse; α too high → uniform but poor specialisation")
print("  4. Capacity factor CF=1.25 absorbs ~20% imbalance without token dropping")
print("  5. MoE decouples total params (memory) from active params (compute)")
print("  6. Expert specialisation emerges naturally from routing gradient")
print("  7. All-to-all communication can be overlapped with expert compute")
print("  8. Gini coefficient and routing entropy quantify load balance quality")
print("  9. MoE models need N× more memory than dense models at same compute")
print(" 10. Z-loss regularises router logits to prevent near-one-hot collapse")